# ML-09 — Validation and Research Claim Audit

This notebook turns the same careful reading we gave the FlyRank research paper back onto my own Week-5 model. The paper was built for a broad audience and holds itself to disclosed standards; my job here is not to grade it, but to practice the next level of rigor on my own work.

Lens pulled from `skills/README.md` -> `skills/hunting-leakage-and-validating/SKILL.md` (leakage taxonomy, grouped/time-aware splits, base rates) + `skills/flyrank/flyrank-data/SKILL.md` (the data contract).

Order of work: (1) two paper findings + my methodology questions, (2) my model under an honest split (before/after), (3) leakage audit, (4) claim rewrite, (5) self-check.

## 1. Two paper findings + my methodology questions

The paper: *The State of AI-Driven SEO — FlyRank Data Report, April 2026* (public data report). I picked two findings I actually used the same logic on in my own work, so the questions I ask the paper are questions I then have to answer about myself — the way I would want my own model reviewed.

**Finding A — the Freshness Multiplier (Finding 4, CONFIRMED).**
The paper reports that freshness is a strong signal: 31-90 days since update shows the strongest stable growth-to-decline ratio (5.43:1), and separately that pages older than a year refreshed in the last 30 days jumped from ~23 to ~37 health (1.6x) and from 82 to 4.2K impressions (52x) versus old pages last updated 181-360 days ago. It also flags that the 361+ bucket (n=802, 21 declining pages) is too small to read and that 0-30 days "too new to read reliably".

**The methodology question I would ask (label provenance):** where does the "refreshed" label come from, and does the comparison hold up? The paper discloses it detects an update "through internal workflow logic and algorithmic signals, not one fixed edit type" — a deterministic internal flag that mixes title/meta/body/link/UX edits. My respectful question: a cohort of pages a team chose to refresh last month is a *self-selected* group, not a matched control. Teams typically refresh pages they already expect to recover (proven historical visibility, recent flattening) — so "refreshed pages do better" may partly be "we refreshed the pages that were already winning". And because the refresh window (0-30d) overlaps the growth/decline window (also last-30d-vs-prev-30d), the two measurement windows are entangled. The 52x card compares refreshed vs *never-touched* old pages; I would ask whether that comparison was matched on pre-refresh health and impressions before leaning on it.

**Honest echo onto me:** my own Week-4 staleness check hit the same wall. My `days_since_last_update` is clumped at bulk batch dates (20/104/22/8 days — 88% of rows share the top-5 dates), so "stale" is partly "member of a client batch-update cohort" and I cannot separate elapsed time from cohort effects. The cell below shows that clumping so the paper question is not cheap to ask of others and silent about myself.

**Finding B — Keyword drift is not a problem (Finding 12, NEW ANALYSIS).**
The paper claims keyword match share does *not* predict performance: Pearson correlations near zero (impressions -0.039, clicks -0.006, health -0.013), the 5-10% match bucket having the highest average impressions (6.5K), and pages at 0% match averaging only 160 impressions; "drift is not a problem — it's a feature."

**The methodology question I would ask (label + validation design):** how is "on-target" defined, and does the validation support the redesign claim? The on-target share is a string-matching construction over the target keyword and the query history — stemming, plural/tense, and phrase-boundary rules decide which queries count as "matching", and small bucket cells (e.g. 0% match bucket) are discrete counts that shift when the matching rule changes. Second, the correlations are computed on *same-window* aggregates: impressions, clicks and match share all come from the same snapshot, with no held-out set and no confidence intervals (the paper discloses it does not report p-values or CIs; I am not asking it to — I am asking how far the "write broader" guidance can lean on same-window association).

**Honest echo onto me:** my Week-4 signal check made the same measurement move — I correlated CTR against position on the *same* 90-day snapshot, so my honest claim is association, never "changing position causes CTR to change". My Week-5 model shares the same overlap: the label (last-30d vs prev-30d trend) sits inside the feature window (90-day sums). That is exactly what Section 3 of this notebook measures, because a question worth asking the paper is a question worth asking myself first.

Back the paper questions with my own data (the two echoes, concretely): batch-date clumping of `days_since_last_update` (the label-provenance wall from Finding A) and the same-snapshot nature of every signal (the validation-design note from Finding B). Public-safe: only aggregate counts and pseudonyms.

In [1]:
# The two echoes, on my own snapshot --------------------------------------
import os
from pathlib import Path

import pandas as pd

ROOT = Path(os.getcwd()).resolve()
for _ in range(6):
    if (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        break
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

print("--- echo A: is fresh/stale partly a batch-cohort artifact? (the label-provenance question) ---")
print(df["days_since_last_update"].value_counts().head(5).to_string())
top5 = set(df["days_since_last_update"].value_counts().head(5).index)
print(f"share of all rows in the top-5 bulk dates: {100 * df['days_since_last_update'].isin(top5).mean():.0f}%")
print("-> a 'stale' page is often a member of a client batch-update cohort; the paper's question is mine too.")

print()
print("--- echo B: every signal I compare lives in ONE snapshot (the validation-design question) ---")
cols = list(df.columns)
recent = [c for c in cols if "_last_30d" in c or "_prev_30d" in c]
print("same-window siblings present but excluded from every model feature set:", recent)
print("label derived from last-30d vs prev-30d -> any 90d sum in features overlaps that label window.")
print("-> so any cross-signal reading on this data is an observed association, not a causal ranking rule.")

--- echo A: is fresh/stale partly a batch-cohort artifact? (the label-provenance question) ---
days_since_last_update
20     11573
104     8773
22      3564
8       1929
13       515
share of all rows in the top-5 bulk dates: 88%
-> a 'stale' page is often a member of a client batch-update cohort; the paper's question is mine too.

--- echo B: every signal I compare lives in ONE snapshot (the validation-design question) ---
same-window siblings present but excluded from every model feature set: ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
label derived from last-30d vs prev-30d -> any 90d sum in features overlaps that label window.
-> so any cross-signal reading on this data is an observed association, not a causal ranking rule.


## 2. My model under an honest split (before/after)

The Week-5 pipeline used a single `GroupShuffleSplit` holdout by `client_id`. The improvement shown here is the honest **before/after**: I re-run the *identical* Week-5 pipeline twice — first under a plain random split (memorization allowed), then under the client-grouped split (a client never seen in training). The gap between the two numbers *is* the finding about memorization.

Note on the time-aware option: this anonymized release is a **single snapshot** — every row shares the same trailing-90-day window, so there is no earlier period to train on and no later period to test on. A true temporal split is not representable. The equivalent time awareness lives in Section 3, where I audit feature-window overlap against the label window.

Identical Week-5 code (features, label, models, metrics), re-run twice. The label `is_declining_label = (trend_direction == "down")` is evidence, never a feature.

In [2]:
# --- 1. data, seed, versions (identical to Week 5) -------------------------
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42

ROOT = Path(os.getcwd()).resolve()
for _ in range(6):
    if (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        break
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

import sklearn as sk

# --- features: explicit, honest, single snapshot (no label-adjacent columns) ---
NUM_COUNT = ["search_volume", "cpc", "word_count", "char_count",
             "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
             "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d"]   # log1p: heavy tail
NUM_RAW = ["competition", "days_with_impressions", "days_with_sessions",
           "content_age_days", "days_since_last_update", "ctr", "avg_position",
           "engagement_rate", "scroll_rate", "ai_traffic_pct"]                 # already scaled/bounded
TIERS = ["competition_level", "age_tier", "freshness_tier",
         "word_count_tier", "impression_tier", "position_tier"]                # ordinal codes
NOMINAL = ["content_type", "main_intent"]                                      # one-hot

X = pd.DataFrame(index=df.index)
for c in NUM_COUNT:
    X["log_" + c] = np.log1p(pd.to_numeric(df[c], errors="coerce")).fillna(0).astype(float)
for c in NUM_RAW:
    X[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(float)
for c in TIERS:
    X[c] = df[c].astype("category").cat.codes.astype(int)
for c in NOMINAL:
    X = X.join(pd.get_dummies(df[c].fillna("unknown"), prefix=c, dtype=int).astype(int))

y = df["is_declining_label"].astype(int)
print(f"rows={len(df):,}  features={X.shape[1]}  sklearn={sk.__version__}  numpy={np.__version__}  pandas={pd.__version__}  label share={y.mean():.3f}  seed={SEED}")

# --- model + metrics (identical to Week 5) -----------------------------------
def week4_rule_score(imp, dsul, pos, ctr):
    stale = ((dsul >= 180) & (imp >= 300)).astype(int)
    gap = ((pos > 0) & (pos <= 10) & (ctr < 0.5) & (imp >= 300)).astype(int)
    return np.log1p(imp) * (1 + stale) * (1 + gap)

def precision_at_k(y_true, score, ks=(10, 20, 50, 100)):
    order = np.argsort(-np.asarray(score))
    yy = np.asarray(y_true)
    return [round(float(yy[order[:k]].mean()), 3) for k in ks]

def eval_row(name, score, yy):
    return [name] + precision_at_k(yy, score) + [round(roc_auc_score(yy, score), 3)]

def fresh_models():
    return {
        "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
        "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
        "gradient_boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=5, random_state=SEED),
    }

def run_holdout_trte(tr, te, label):
    Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
    base = week4_rule_score(df["impressions_90d"].iloc[te].values, df["days_since_last_update"].iloc[te].values,
                            df["avg_position"].iloc[te].values, df["ctr"].iloc[te].values)
    probs = {"week4_rule_baseline": base}
    for name, m in fresh_models().items():
        m.fit(Xtr, ytr)
        probs[name] = m.predict_proba(Xte)[:, 1]
    tbl = pd.DataFrame([["base_rate"] + [None] * 4 + [round(float(yte.mean()), 3)]]
                       + [eval_row(n, p, yte) for n, p in probs.items()],
                       columns=["method", "P@10", "P@20", "P@50", "P@100", "AUC"])
    print(f"{label}  test n={len(yte):,}  base rate={yte.mean():.3f}")
    print(tbl.to_string(index=False))
    print()
    return {"label": label, "table": tbl, "probs": probs, "yte": yte, "te": te}

# --- BEFORE: random split (memorization allowed) -------------------------------
tr_r, te_r = train_test_split(np.arange(len(X)), test_size=0.20, random_state=SEED)
print("=== BEFORE: random split (a client's pages on BOTH sides) ===")
before = run_holdout_trte(tr_r, te_r, "random split:")

# --- AFTER: client-grouped split (a client never seen in training) ------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_g, te_g = next(gss.split(X, y, groups=df["client_id"]))
print("=== AFTER: client-grouped split (the Week-5 honest split) ===")
after = run_holdout_trte(tr_g, te_g, "grouped split:")

print("=== before/after on the SAME methods (the gap = memorization removed) ===")
side = pd.DataFrame({
    "random": before["table"].set_index("method")["AUC"],
    "grouped": after["table"].set_index("method")["AUC"],
})
side["drop"] = (side["random"] - side["grouped"]).round(3)
print(side.round(3).to_string())

# --- the same lesson as repeated K-folds ---------------------------------------
def kfold_readout(splitter, name, n_to_print=1):
    aucs = []
    for trk, tek in splitter.split(X, y, groups=df["client_id"]) if name == "grouped" else splitter.split(X, y):
        hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=5, random_state=SEED)
        hgb.fit(X.iloc[trk], y.iloc[trk])
        aucs.append(roc_auc_score(y.iloc[tek], hgb.predict_proba(X.iloc[tek])[:, 1]))
    return np.mean(aucs), np.std(aucs)

gf, gs = kfold_readout(GroupKFold(n_splits=5), "grouped")
kf, ks = kfold_readout(KFold(n_splits=5, shuffle=True, random_state=SEED), "random")
print(f"\nKFold(5) HGB AUC  (random folds):  {kf:.3f} +- {ks:.3f}")
print(f"GroupKFold(5) HGB AUC (client):    {gf:.3f} +- {gs:.3f}")

rows=30,000  features=36  sklearn=1.9.0  numpy=2.5.1  pandas=3.0.3  label share=0.542  seed=42
=== BEFORE: random split (a client's pages on BOTH sides) ===


random split:  test n=6,000  base rate=0.545
             method  P@10  P@20  P@50  P@100   AUC
          base_rate   NaN   NaN   NaN    NaN 0.545
week4_rule_baseline   0.6  0.50  0.42   0.44 0.596
logistic_regression   1.0  0.90  0.90   0.91 0.716
      random_forest   1.0  1.00  0.98   0.95 0.782
  gradient_boosting   0.9  0.95  0.96   0.95 0.786

=== AFTER: client-grouped split (the Week-5 honest split) ===


grouped split:  test n=6,163  base rate=0.511
             method  P@10  P@20  P@50  P@100   AUC
          base_rate   NaN   NaN   NaN    NaN 0.511
week4_rule_baseline   0.4  0.40  0.42   0.41 0.542
logistic_regression   0.8  0.75  0.76   0.72 0.622
      random_forest   0.6  0.75  0.68   0.69 0.610
  gradient_boosting   0.9  0.90  0.90   0.86 0.622

=== before/after on the SAME methods (the gap = memorization removed) ===
                     random  grouped   drop
method                                     
base_rate             0.545    0.511  0.034
week4_rule_baseline   0.596    0.542  0.054
logistic_regression   0.716    0.622  0.094
random_forest         0.782    0.610  0.172
gradient_boosting     0.786    0.622  0.164



KFold(5) HGB AUC  (random folds):  0.778 +- 0.004
GroupKFold(5) HGB AUC (client):    0.684 +- 0.043


**What the numbers say.** Same model code, same seed, same 20% test size. The **random split** lets pages from the same client appear on both sides, so the model can memorise client-specific quirks; moving to the **client-grouped split** removes that crutch and the honest numbers drop. The size of the drop is the amount of memorisation the grouped split removes. Week 5 already reported the grouped numbers (P@10 0.90 / AUC 0.622 for gradient boosting); this cell recomputes them in the same run so before/after sit on one page, and adds the K-fold version of the same lesson.

#### Where the honest model is wrong (real failure examples)

Ranking pages for *review* is decision support, so both error types cost differently. Public-safe checks only: pseudonym IDs, no client names or URLs.

In [3]:
# --- real failure examples, on the honest (grouped) test split -------------
test = df.iloc[after["te"]].copy()
test["proba"] = after["probs"]["gradient_boosting"]
test["predicted"] = (test["proba"] >= 0.5).astype(int)

fn = test[(test["predicted"] == 0) & (test["is_declining_label"] == 1)]
fp = test[(test["predicted"] == 1) & (test["is_declining_label"] == 0)]
sad = test[(test["predicted"] == 0) & (test["is_declining_label"] == 0)]

print(f"false negatives: {len(fn):,} (predicted review-later, already-observed declining)")
print(f"  median impressions {int(fn['impressions_90d'].median()):,} | share with impressions<500 {100*fn['impressions_90d'].lt(500).mean():.0f}%")
print(f"false positives:   {len(fp):,} (predicted review-now, not observed declining)")
print(f"  median impressions {int(fp['impressions_90d'].median()):,} | median position {fp['avg_position'].median():.1f} | median days_since_last_update {int(fp['days_since_last_update'].median())}")
cols = ["content_id", "proba", "is_declining_label", "impressions_90d", "days_with_impressions",
        "avg_position", "ctr", "days_since_last_update", "content_age_days", "trend_direction", "content_type"]
print("\n3 concrete wrong rows (public-safe: pseudonyms only):")
print(fn.sort_values("impressions_90d", ascending=False)[cols].head(2).to_string(index=False))
print(fp[cols].head(1).to_string(index=False))
print("\nshape of the honest decision: a review QUEUE (ranked), not an automated action.")

false negatives: 1,122 (predicted review-later, already-observed declining)
  median impressions 334 | share with impressions<500 53%
false positives:   1,415 (predicted review-now, not observed declining)
  median impressions 787 | median position 10.5 | median days_since_last_update 20

3 concrete wrong rows (public-safe: pseudonyms only):
          content_id    proba  is_declining_label  impressions_90d  days_with_impressions  avg_position  ctr  days_since_last_update  content_age_days trend_direction    content_type
content_8c19996aa890 0.274215                   1           509252                     88           2.5 0.15                      20               445            down keyword article
content_4c36c775b818 0.310062                   1           463103                     88           2.3 0.41                      20               445            down keyword article
          content_id    proba  is_declining_label  impressions_90d  days_with_impressions  avg_position  ct

## 3. Leakage audit

The same hunt from the skill checklist, on my final feature set, against all three leakage categories.

Timeline drawn honestly (single snapshot):

```
snapshot date = export date (all rows share the same trailing window)
feature windows (knowable at decision time):
   - 90-day sum columns (impressions/clicks/sessions/..._90d)   [IN features]
   - state columns (avg_position, ctr, engagement, ages, tiers) [IN features]
   - keyword-level (search_volume, cpc, competition)            [IN features]
label windows (the thing I predict):
   - trend_direction = last-30d vs prev-30d comparison          [INSIDE the 90d sums]
sibling columns that MUST stay out: *_last_30d, *_prev_30d      [excluded in Week 5; blocklist re-checked below]
```

That means three probes below:
- **Probe A — overlapping-window sums.** The 8 `*_90d` sums contain the last-30/prev-30 days the label is computed from. Strictly, they are overlap, so I measure how much score they add: model WITH the sums vs model WITHOUT (state + keyword features only). Honest framing: any residual signal is association, not forecast.
- **Probe B — decision-derived / product-flag.** The Week-4 rule score is *my own* product flag. Using it as a feature would mean learning my old rule, not the world — circular. I show the number, then keep it out (baseline to beat, never an input).
- **Probe C — harness sanity.** Deliberately add a *known*-leaky sibling column (`impressions_last_30d`) and watch the score jump. If it does not jump, the test harness itself is broken; it should, and then I remove it.

Three targeted probes plus the blocklist re-check, all on the same client-grouped split so every number sits next to the honest baseline (from Section 2).

In [4]:
# --- blocklist re-check (leakage category 1: label-derived / sibling) -------
blocked = ["trend_direction", "trend_pct", "client_id", "content_id"]
recent_sib = [c for c in df.columns if "_last_30d" in c or "_prev_30d" in c]
in_feats = sorted(set(X.columns) & (set(blocked) | set(recent_sib)))
print("blocklist/sibling columns present in my feature set:", in_feats if in_feats else "NONE (clean)")

overlap_sums = [c for c in NUM_COUNT if c.endswith("_90d")]
print("overlap disclosure: 90d sums in features that CONTAIN the label windows:", overlap_sums)

# --- common setup: the grouped split from Section 2 ---------------------------
Xtr, Xte, ytr, yte = X.iloc[tr_g], X.iloc[te_g], y.iloc[tr_g], y.iloc[te_g]

# Probe A: overlapping 90d sums removed (state + keyword features only)
drop = [c for c in NUM_COUNT if c.endswith("_90d")]
X_no_sum = X.drop(columns=[f"log_{c}" for c in drop])
mA = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=5, random_state=SEED)
mA.fit(X_no_sum.iloc[tr_g], y.iloc[tr_g])
auc_no_sum = roc_auc_score(yte, mA.predict_proba(X_no_sum.iloc[te_g])[:, 1])

# Probe B: my own Week-4 rule score as a feature (product flag, circular)
X_rule = X.copy()
rule_all = week4_rule_score(df["impressions_90d"].values, df["days_since_last_update"].values,
                            df["avg_position"].values, df["ctr"].values)
X_rule["week4_rule_score"] = rule_all
mB = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=5, random_state=SEED)
mB.fit(X_rule.iloc[tr_g], y.iloc[tr_g])
auc_rule_feat = roc_auc_score(yte, mB.predict_proba(X_rule.iloc[te_g])[:, 1])

# Probe C: harness sanity — deliberately add a leaky sibling column
X_leak = X.copy()
X_leak["impressions_last_30d"] = pd.to_numeric(df["impressions_last_30d"], errors="coerce").fillna(0)
mC = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=5, random_state=SEED)
mC.fit(X_leak.iloc[tr_g], y.iloc[tr_g])
auc_leak = roc_auc_score(yte, mC.predict_proba(X_leak.iloc[te_g])[:, 1])

print("\n===== leakage probes (grouped split, same 6,163 test rows; AUC) =====")
print(f"honest features (as shipped, Week 5).... {after['table'].set_index('method').loc['gradient_boosting','AUC']:.3f}")
print(f"Probe A  no overlapping 90d sums........ {auc_no_sum:.3f}   (overlap removed)")
print(f"Probe B  + week4_rule_score as feature.. {auc_rule_feat:.3f}   (product-flag, circular, dropped)")
print(f"Probe C  + impressions_last_30d (LEAK).. {auc_leak:.3f}   (harness sanity: should jump)")

blocklist/sibling columns present in my feature set: NONE (clean)
overlap disclosure: 90d sums in features that CONTAIN the label windows: ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']



===== leakage probes (grouped split, same 6,163 test rows; AUC) =====
honest features (as shipped, Week 5).... 0.622
Probe A  no overlapping 90d sums........ 0.621   (overlap removed)
Probe B  + week4_rule_score as feature.. 0.624   (product-flag, circular, dropped)
Probe C  + impressions_last_30d (LEAK).. 0.849   (harness sanity: should jump)


**What each probe confessed.** Probe A removed all eight overlapping 90-day sums and the honest AUC did not move (0.622 -> 0.621): the model does not lean on the label-overlapping windows to earn its score — the state and keyword features alone carry it. Probe B added my own Week-4 rule flag as a feature and the score did not inflate (0.622 -> 0.624): the model is not secretly my hand-rule; the rule is kept out as a baseline-to-beat regardless. Probe C then proved the blocklist is load-bearing: one sibling column (`impressions_last_30d`) that overlaps the label windows jumps AUC from 0.622 to 0.849 — the leakage warning is real here even though the shipped 90-day sums barely overlap in practice. Net: no leak sits in this pipeline, and the claim this model is allowed to make is about already-observed state — never a forecast.

In [5]:
# verdict line, printed so the read lives next to the numbers
print(f"Verdict (safe language): no leak sits in the shipped features. Probe C proves the blocklist is ")
print(f"load-bearing: one sibling column (impressions_last_30d) overlapping the label windows jumps AUC ")
print(f"from {after['table'].set_index('method').loc['gradient_boosting','AUC']:.3f} to {auc_leak:.3f}. ")
print(f"Probe A (drop the eight overlapping 90d sums -> {auc_no_sum:.3f}) and Probe B (add my own rule flag ")
print(f"-> {auc_rule_feat:.3f}) barely move the score: the model leans on state and keyword features, not ")
print(f"the label-overlapping sums, and it is not my old rule. Claim scope: association with already-observed ")
print(f"decline, decision-support only, never a forecast.")

Verdict (safe language): no leak sits in the shipped features. Probe C proves the blocklist is 
load-bearing: one sibling column (impressions_last_30d) overlapping the label windows jumps AUC 
from 0.622 to 0.849. 
Probe A (drop the eight overlapping 90d sums -> 0.621) and Probe B (add my own rule flag 
-> 0.624) barely move the score: the model leans on state and keyword features, not 
the label-overlapping sums, and it is not my old rule. Claim scope: association with already-observed 
decline, decision-support only, never a forecast.


## 4. Claim rewrite

Below is Week-5's boldest sentence, then the same sentence in safe language (observed / measured / directional / decision-support). The words changed, not the evidence.

| | Claim |
|---|---|
| **Week-5, bolded** | "HGB tops the queue of pages to review at P@10 = 0.90 on held-out clients, and refresh actions should be ordered by its probability." |
| **Rewritten** | "**Observed** on the one anonymized snapshot: across a client-grouped holdout (held-out clients, test n=6,163), the gradient-boosting ranker **measured** a higher rate of already-observed decline at the top of the review queue (P@10 = 0.90) than the hand-built baseline (P@10 = 0.40) or the ~51% base rate. This is **directional, decision-support** evidence for *which pages to review first* — it is an **association** with a page's current declining profile, not a forecast of future performance, not a claim that refreshing those pages causes recovery, and not a claim about a new client's backlog before it is actually scoped." |

Why the rewrite is not just wording: (1) the label is *already-observed* trend, and my features share overlapping windows, so "will decline" would overclaim a state estimate; (2) P@10 is a ranking metric on one snapshot, not a guarantee on the next month's snapshot; (3) a queue for human review is decision support — the final action stays with the editor.

In [6]:
# claim rewrite, rendered as the table from Section 4
import textwrap
rows = [
    ("Week-5 star sentence (bold of it):",
     "HGB tops the queue of pages to review at P@10 = 0.90 on held-out clients; order refresh actions by its probability."),
    ("Safe-language rewrite:",
     "Observed on the single anonymized snapshot, client-grouped holdout (held-out clients, test n=6,163): the gradient-boosting "
     "ranker measured a higher rate of already-observed decline at the top of the review queue (P@10 = 0.90) than the hand-built "
     "baseline (P@10 = 0.40) and the ~51% base rate. This is directional, decision-support evidence for review order: an association "
     "with a page's current declining profile, not a forecast, not a refresh-causes-recovery claim, not a claim about a new client."),
]
for label, txt in rows:
    print(label)
    print("  " + "\n  ".join(textwrap.wrap(txt, 96)))
    print()

Week-5 star sentence (bold of it):
  HGB tops the queue of pages to review at P@10 = 0.90 on held-out clients; order refresh actions
  by its probability.

Safe-language rewrite:
  Observed on the single anonymized snapshot, client-grouped holdout (held-out clients, test
  n=6,163): the gradient-boosting ranker measured a higher rate of already-observed decline at the
  top of the review queue (P@10 = 0.90) than the hand-built baseline (P@10 = 0.40) and the ~51%
  base rate. This is directional, decision-support evidence for review order: an association with
  a page's current declining profile, not a forecast, not a refresh-causes-recovery claim, not a
  claim about a new client.



## Self-check

Before submitting, honest confirmations:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed in place via nbconvert, `exit_status` 0)
- [x] No client names, URLs, or private queries anywhere — only pseudonyms and aggregates
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Two paper findings named with a concrete methodology question each, framed constructively
- [x] Own model re-run under random vs client-grouped split, before/after on one page
- [x] Leakage audit covers all three categories (label-derived, overlapping window, product flag) with a harness-sanity probe
- [x] Real failure examples shown
- [x] Committed to the repo under `work/notebooks/`